# Task 7 Colab speed test

Measures how fast the SCIL training loops (SQIL, IQ-Learn, GAIL, evaluation) run on this Colab GPU, so we can decide whether to move a batch of Task 7 runs here. Nothing is trained for real; stand-in data are two untrained-policy episodes.

**Before running**
1. Put `task7_colab_bench.zip` in Google Drive at `MyDrive/task7_colab_bench/`.
2. Runtime → Change runtime type → choose the GPU (A100 first, then L4 for comparison) and turn on **High-RAM**.
3. Note the GPU's compute-unit rate in the Resources panel (CU/hour) and enter it in the last cell.
4. Runtime → Run all. The full suite takes about 20–40 minutes.
5. When it finishes: **Runtime → Disconnect and delete runtime**, otherwise it keeps using compute units.

Results are written to `MyDrive/task7_colab_bench/results/<gpu>_<time>/`.

In [ ]:
# 1. Hardware
!nvidia-smi --query-gpu=name,memory.total --format=csv
!nproc; free -g | head -2
import time; T_START = time.time()

In [ ]:
# 2. Get the code from Drive
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/task7_colab_bench'
!rm -rf /content/task7_colab_bench && unzip -q "$DRIVE/task7_colab_bench.zip" -d /content && head -3 /content/task7_colab_bench/MANIFEST.txt

In [ ]:
# 3. Install dependencies (keeps Colab's own torch / numpy). ogbench without dm_control/labmaze, ananke without old numpy pins
!pip -q install -r /content/task7_colab_bench/bench/requirements_colab.txt
!pip -q install --no-deps ogbench==1.2.1 ananke-causal==0.5.0
import os, sys
os.environ['PYTHONPATH'] = '/content/task7_colab_bench/code'
os.environ['MUJOCO_GL'] = 'egl'
os.environ['PYTHONWARNINGS'] = 'ignore'
!cd /content/task7_colab_bench && python -c "import torch, numpy, causal_gym, causal_rl; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'numpy', numpy.__version__)"

In [ ]:
# 4. Smoke test: one short SQIL measurement on the GPU (about 1 minute)
!cd /content/task7_colab_bench && python bench/bench_task7.py antmed sqil causal --device cuda --updates 50

In [ ]:
# 5. Full suite: single runs of every algorithm, then 4 and 8 runs at once sharing this GPU
import torch, datetime
gpu = torch.cuda.get_device_name(0).replace(' ', '_')
OUT = f"{DRIVE}/results/{gpu}_{datetime.datetime.now():%Y%m%d_%H%M}"
print('results ->', OUT)
!cd /content/task7_colab_bench && python bench/run_suite.py --device cuda --out "$OUT" --parallel 1,4,8

In [ ]:
# 6. Summary. Fill in the CU/hour shown in the Resources panel for this GPU.
CU_PER_HOUR = 0.0   # e.g. the A100 rate shown in Colab
import psutil, subprocess
RAM_GB = psutil.virtual_memory().total / 2**30
subprocess.run([sys.executable, "bench/summarize_bench.py", OUT, "--cu-per-hour", str(CU_PER_HOUR), "--ram-gb", f"{RAM_GB:.0f}"],
               cwd="/content/task7_colab_bench", check=False)
print(f"notebook time so far: {(time.time() - T_START) / 60:.0f} min")


**Done.** Send the summary output (or the `results` folder) back. Then Runtime → Disconnect and delete runtime.